# Phase I — SegFormer-B1 road masks

Hierarchical transformer segmenter (NVIDIA MiT-B1 backbone). Same aug + loss as DeepLab for fair comparison.

Outputs: `best_road_model_segformer_b1.pth`, `outputs/masks_segformer/`


## Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Install packages

In [ ]:
import subprocess
subprocess.run([
    "pip", "install", "-q",
    "transformers", "accelerate", "albumentations", "opencv-python", "tqdm",
    "segmentation-models-pytorch",
], check=True)
print('ok')


## Paths & split

In [ ]:
import os, random

DRIVE_BASE = "/content/drive/MyDrive/RouteResilience"
SAVE_DIR = f"{DRIVE_BASE}/checkpoints"
DATA_DIR = f"{DRIVE_BASE}/datasets/train"
MASK_DIR = f"{DRIVE_BASE}/outputs/masks_segformer"
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(MASK_DIR, exist_ok=True)

CKPT_PATH = f"{SAVE_DIR}/best_road_model_segformer_b1.pth"
MODEL_TAG = 'segformer_b1'

IMG_SIZE = 512
BATCH_SIZE = 4  # SegFormer-B1 needs more VRAM — bump to 6 if T4 allows
EPOCHS = 30
LR = 6e-5  # slightly lower LR works well for transformers

all_files = os.listdir(DATA_DIR)
sat = {f.replace('_sat.jpg', '') for f in all_files if f.endswith('_sat.jpg')}
mask = {f.replace('_mask.png', '') for f in all_files if f.endswith('_mask.png')}
all_ids = sorted(sat & mask)
random.seed(42)
random.shuffle(all_ids)
split = int(0.8 * len(all_ids))
train_ids, val_ids = all_ids[:split], all_ids[split:]
print(f"Train: {len(train_ids)} | Val: {len(val_ids)}")
print(f"Checkpoint: {CKPT_PATH}")


## Dataset

In [ ]:
import cv2
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2

def get_train_transforms(img_size=512):
    return A.Compose([
        A.RandomRotate90(p=0.5),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.3),
        A.Resize(img_size, img_size),
        A.RandomShadow(
            shadow_roi=(0, 0, 1, 1),
            num_shadows_limit=(1, 3),
            shadow_intensity_range=(0.3, 0.7),
            p=0.5,
        ),
        A.CoarseDropout(
            num_holes_range=(1, 8),
            hole_height_range=(16, 48),
            hole_width_range=(16, 48),
            fill=0,
            p=0.5,
        ),
        A.RandomBrightnessContrast(p=0.4),
        A.HueSaturationValue(p=0.3),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])

def get_val_transforms(img_size=512):
    return A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])

class DeepGlobeDataset(Dataset):
    def __init__(self, data_dir, ids, transform=None):
        self.data_dir = data_dir
        self.ids = ids
        self.transform = transform

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        img_id = self.ids[idx]
        img = cv2.cvtColor(
            cv2.imread(f"{self.data_dir}/{img_id}_sat.jpg"), cv2.COLOR_BGR2RGB
        )
        mrgb = cv2.cvtColor(
            cv2.imread(f"{self.data_dir}/{img_id}_mask.png"), cv2.COLOR_BGR2RGB
        )
        mask = (mrgb[:, :, 0] > 127).astype(np.float32)
        if self.transform:
            aug = self.transform(image=img, mask=mask)
            img, mask = aug["image"], aug["mask"]
        return img, mask.unsqueeze(0)

print(f"dataset: {len(train_ids)} train, {len(val_ids)} val")


## Model

SegFormer with MiT-B1 encoder from Hugging Face (`nvidia/mit-b1`).

In [ ]:
from transformers import SegformerForSemanticSegmentation

def build_model():
    model = SegformerForSemanticSegmentation.from_pretrained(
        "nvidia/mit-b1",
        num_labels=1,
        ignore_mismatched_sizes=True,
    )
    n = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"SegFormer-B1 (MiT-B1) | Params: {n:,}")
    return model

import segmentation_models_pytorch as smp

class CombinedLoss(torch.nn.Module):
    def __init__(self, alpha=0.4):
        super().__init__()
        self.alpha = alpha
        self.bce = smp.losses.SoftBCEWithLogitsLoss()
        self.dice = smp.losses.DiceLoss(mode="binary", from_logits=True)

    def forward(self, preds, targets):
        return self.alpha * self.bce(preds, targets) + (1 - self.alpha) * self.dice(
            preds, targets
        )

def calculate_metrics(preds, targets, threshold=0.5):
    preds = (torch.sigmoid(preds) > threshold).float()
    targets = targets.float()
    inter = (preds * targets).sum(dim=(1, 2, 3))
    union = preds.sum(dim=(1, 2, 3)) + targets.sum(dim=(1, 2, 3))
    dice = (2.0 * inter + 1e-6) / (union + 1e-6)
    iou = (inter + 1e-6) / (union - inter + 1e-6)
    return iou.mean().item(), dice.mean().item()


## Training

In [ ]:
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
from tqdm import tqdm
import matplotlib.pyplot as plt
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

train_loader = DataLoader(
    DeepGlobeDataset(DATA_DIR, train_ids, get_train_transforms(IMG_SIZE)),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
)
val_loader = DataLoader(
    DeepGlobeDataset(DATA_DIR, val_ids, get_val_transforms(IMG_SIZE)),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)

model = build_model().to(device)
loss_fn = CombinedLoss(0.4)
optimizer = optim.Adam(model.parameters(), lr=LR)
scheduler = ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=5)

best_val_iou = 0.0
history = {"train_loss": [], "val_loss": [], "train_iou": [], "val_iou": []}

def forward_logits(model, images):
    out = model(pixel_values=images).logits
    out = F.interpolate(out, size=(IMG_SIZE, IMG_SIZE), mode='bilinear', align_corners=False)
    return out

print(f"\nStarting training for {EPOCHS} epochs on {device}...\n")
for epoch in range(1, EPOCHS + 1):
    model.train()
    t_loss = t_iou = 0.0
    for images, masks in tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS} [Train]"):
        images = images.to(device, torch.float32)
        masks = masks.to(device, torch.float32)
        optimizer.zero_grad()
        outputs = forward_logits(model, images)
        loss = loss_fn(outputs, masks)
        loss.backward()
        optimizer.step()
        t_loss += loss.item()
        iou, _ = calculate_metrics(outputs, masks)
        t_iou += iou

    model.eval()
    v_loss = v_iou = 0.0
    with torch.no_grad():
        for images, masks in tqdm(val_loader, desc=f"Epoch {epoch}/{EPOCHS} [Val]"):
            images = images.to(device, torch.float32)
            masks = masks.to(device, torch.float32)
            outputs = forward_logits(model, images)
            loss = loss_fn(outputs, masks)
            v_loss += loss.item()
            iou, _ = calculate_metrics(outputs, masks)
            v_iou += iou

    t_loss /= len(train_loader)
    t_iou /= len(train_loader)
    v_loss /= len(val_loader)
    v_iou /= len(val_loader)
    scheduler.step(v_loss)

    history["train_loss"].append(t_loss)
    history["val_loss"].append(v_loss)
    history["train_iou"].append(t_iou)
    history["val_iou"].append(v_iou)

    print(
        f"Epoch {epoch:02d} | Train Loss: {t_loss:.4f} IoU: {t_iou:.4f} | "
        f"Val Loss: {v_loss:.4f} IoU: {v_iou:.4f}"
    )
    if v_iou > best_val_iou:
        best_val_iou = v_iou
        torch.save(model.state_dict(), CKPT_PATH)
        print(f"  new best val iou {v_iou:.4f}")

print(f"\nfinished. best val iou {best_val_iou:.4f}")
print(f"Checkpoint: {CKPT_PATH}")


## Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history["train_loss"], label="Train Loss", color="tomato")
axes[0].plot(history["val_loss"], label="Val Loss", color="steelblue")
axes[0].set_title("Loss Curves")
axes[0].legend(); axes[0].grid(True)
axes[1].plot(history["train_iou"], label="Train IoU", color="tomato")
axes[1].plot(history["val_iou"], label="Val IoU", color="steelblue")
axes[1].set_title("IoU Score Curves")
axes[1].legend(); axes[1].grid(True)
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/training_curves_{MODEL_TAG}.png", dpi=120)
plt.show()
print(f"Best Val IoU: {best_val_iou:.4f}")


## Inference

In [ ]:
from transformers import SegformerForSemanticSegmentation
import torch.nn.functional as F

model = SegformerForSemanticSegmentation.from_pretrained(
    "nvidia/mit-b1", num_labels=1, ignore_mismatched_sizes=True,
)
model.load_state_dict(torch.load(CKPT_PATH, map_location=device))
model = model.to(device).eval()
print("Loaded", CKPT_PATH)

@torch.no_grad()
def predict_and_save(img_id):
    img = cv2.cvtColor(cv2.imread(f"{DATA_DIR}/{img_id}_sat.jpg"), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    aug = get_val_transforms(IMG_SIZE)(image=img, mask=np.zeros(img.shape[:2], dtype=np.float32))
    t = aug["image"].unsqueeze(0).to(device)
    logits = model(pixel_values=t).logits
    logits = F.interpolate(logits, size=(IMG_SIZE, IMG_SIZE), mode="bilinear", align_corners=False)
    pred = (torch.sigmoid(logits) > 0.5).squeeze().cpu().numpy().astype(np.uint8) * 255
    if pred.shape != (h, w):
        pred = cv2.resize(pred, (w, h), interpolation=cv2.INTER_NEAREST)
    out_path = f"{MASK_DIR}/{img_id}_pred.png"
    cv2.imwrite(out_path, pred)
    return pred

DEMO_IDS = ["493626", "477671", "422265", "194764"]
demo_ids = [i for i in DEMO_IDS if i in val_ids or i in train_ids]
if not demo_ids:
    demo_ids = val_ids[:4]

for img_id in demo_ids:
    predict_and_save(img_id)
    print(f"Saved mask: {MASK_DIR}/{img_id}_pred.png")

print("Done. Masks in", MASK_DIR)


## Quick visual check

In [ ]:
sample_ids = demo_ids[:4]
fig, axes = plt.subplots(len(sample_ids), 3, figsize=(12, 4 * len(sample_ids)))
if len(sample_ids) == 1:
    axes = axes.reshape(1, -1)
for row, img_id in enumerate(sample_ids):
    sat = cv2.cvtColor(cv2.imread(f"{DATA_DIR}/{img_id}_sat.jpg"), cv2.COLOR_BGR2RGB)
    gt = cv2.cvtColor(cv2.imread(f"{DATA_DIR}/{img_id}_mask.png"), cv2.COLOR_BGR2RGB)
    pred = cv2.imread(f"{MASK_DIR}/{img_id}_pred.png", cv2.IMREAD_GRAYSCALE)
    axes[row, 0].imshow(sat); axes[row, 0].set_title(f"{img_id} satellite"); axes[row, 0].axis("off")
    axes[row, 1].imshow(gt); axes[row, 1].set_title("Ground truth"); axes[row, 1].axis("off")
    axes[row, 2].imshow(pred, cmap="gray"); axes[row, 2].set_title("Prediction"); axes[row, 2].axis("off")
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/predictions_{MODEL_TAG}.png", dpi=120)
plt.show()
